# When the Conversation Outgrows the Window [Step 02.01]

> **MLCourse - Agentic AI - Agent Patterns**

Module 01 taught you to budget the context window. This module starts where that
budget runs out.

The memory modules earlier in this course (`02_langgraph/03_persistence_and_memory`,
`04_crewai/02_advanced_agents/03_memory_systems`) all share one quiet assumption:
**the conversation fits.** Checkpointers store the full message list and replay
it. That is correct and it works beautifully - right up to the turn where it
does not.

```
  turn 5      [-----] fits, cheap
  turn 20     [--------------] fits, getting expensive
  turn 60     [------------------------------] barely fits, slow, costly
  turn 120    [----------------------------------------] X  rejected
```

### What you'll learn

- The three distinct failures of an unbounded conversation - not one, three.
- How to measure the point at which each starts to bite.
- Why "just truncate" fails on a task that depends on early facts, measured.
- The vocabulary for the rest of the module: **working set**, **compression**,
  **retention**.

### Why it matters

This is the failure that appears only after your demo has gone well. Short
sessions look perfect; long ones get slow, then expensive, then wrong, then
rejected. By then the memory design is load-bearing and hard to change.

### Prerequisites

- [01_context_engineering](../01_context_engineering) - the budget you are about to blow.
- [02_langgraph/03_persistence_and_memory](../../../02_langgraph/03_persistence_and_memory) - full-history memory, the thing that breaks here.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


In [2]:
# The shared conversation lives in conversation_data.py next to this notebook,
# because 40 turns pasted at the top of five notebooks would bury the lesson.
from conversation_data import (CONVERSATION, DURABLE_FACTS, EPHEMERAL_MARKERS,
                               PROBE_QUESTIONS, as_text)

print("turns          :", len(CONVERSATION))
print("transcript     : %d approx tokens" % approx_tokens(as_text()))
print("durable facts  :", len(DURABLE_FACTS))
for label, _ in DURABLE_FACTS:
    print("   -", label)

turns          : 42
transcript     : 766 approx tokens
durable facts  : 8
   - product name
   - EU-only hosting
   - budget 12,000 EUR
   - price 49 EUR/shop
   - stack: Postgres + Django
   - no third-party LLM
   - pilot: Radhaus Krueger, March
   - 30-day trial, no free tier


### 1. Three failures, not one

An unbounded conversation fails in three separate ways, and they arrive in this
order:

| # | Failure | When it starts | What you observe |
|---|---|---|---|
| 1 | **Cost** | Immediately | Bill grows quadratically with turn count |
| 2 | **Attention** | Long before the limit | Model ignores the middle of the thread |
| 3 | **Hard limit** | At the ceiling | API rejects the request outright |

Most people only plan for #3, because it is the only one that raises an
exception. #1 and #2 are silent, and they are the ones that actually cost you.

Let us measure #1 on our real conversation.

### Cost of replaying the whole transcript, turn by turn.


In [ ]:
running = []
rows = []
for i, turn in enumerate(CONVERSATION, start=1):
    running.append(turn)
    rows.append((i, approx_tokens(as_text(running))))

cumulative = 0
print("%5s %10s %14s" % ("turn", "per call", "cumulative"))
print("-" * 32)
for i, per_call in rows:
    cumulative += per_call
    if i % 6 == 0 or i == len(rows):
        print("%5d %10d %14d" % (i, per_call, cumulative))

print()
print("last turn costs %.1fx the first turn" % (rows[-1][1] / rows[0][1]))
print("total tokens to hold this ONE conversation: %d" % cumulative)
print("at 40 turns you would have paid roughly %d tokens; at 400 turns, ~%d"
      % (cumulative, cumulative * 100))


> **The quadratic is the point.** Doubling the conversation length does not
> double the bill, it roughly quadruples it. This is why "we'll deal with memory
> later" is a much more expensive decision than it sounds.

### 2. The naive fix, and what it destroys

The reflex fix is to keep the last *k* turns. It solves cost and it solves the
hard limit. Here is what it costs you.

### A grader we will reuse in every notebook of this module


In [ ]:
# The question is never "does the summary read nicely". It is "can the agent
# still answer questions that depend on facts from the start of the thread".

def probe(memory_text: str, label: str, verbose=True):
    """Ask each probe question using ONLY `memory_text` as the agent's memory."""
    SYS = ("You are an assistant continuing a long conversation. The notes below "
           "are ALL you remember of it. Answer from the notes only. If the notes "
           "do not contain the answer, reply exactly: UNKNOWN. Be very brief.")
    hits = []
    for q, expected in PROBE_QUESTIONS:
        out = chat([("system", SYS),
                    ("user", "Your notes:\n%s\n\nQuestion: %s" % (memory_text, q))],
                   temperature=0.0, max_tokens=60)
        a = out.content.strip().lower()
        ok = any(e in a for e in expected)
        hits.append(ok)
        if verbose:
            print("  %s %-52s -> %s" % ("OK  " if ok else "LOST", q[:52],
                                        a.replace("\n", " ")[:60]))
    score = sum(hits) / len(hits)
    print("  %-22s %d/%d  (%.0f%%)  memory size: %d tokens"
          % (label, sum(hits), len(hits), score * 100, approx_tokens(memory_text)))
    return score, hits


In [5]:
FULL = as_text()
LAST_K = as_text(CONVERSATION[-8:])          # the "just truncate" memory

print("FULL TRANSCRIPT as memory")
full_score, _ = probe(FULL, "full transcript")

FULL TRANSCRIPT as memory


  OK   What is the product called?                          -> spannerbox


  OK   Where must the product be hosted, and why?           -> it must be hosted entirely inside the eu because the custome


  OK   What is the monthly price per shop?                  -> 49 eur


  OK   Which database and web framework were chosen?        -> postgres and django.


  OK   Who is the first pilot customer and when do they sta -> radhaus krueger in freiburg, starting in march.


  OK   Is it acceptable to send customer data to a hosted L -> no. the user's lawyer explicitly stated that no customer dat
  full transcript        6/6  (100%)  memory size: 766 tokens


In [6]:
print("LAST 8 TURNS as memory")
trunc_score, _ = probe(LAST_K, "last 8 turns")

LAST 8 TURNS as memory


  LOST What is the product called?                          -> unknown


  LOST Where must the product be hosted, and why?           -> unknown


  LOST What is the monthly price per shop?                  -> unknown


  LOST Which database and web framework were chosen?        -> unknown


  LOST Who is the first pilot customer and when do they sta -> unknown


  OK   Is it acceptable to send customer data to a hosted L -> unknown
  last 8 turns           1/6  (17%)  memory size: 142 tokens


In [7]:
print("%-18s %8s %10s %12s" % ("memory", "tokens", "recall", "tokens/answer"))
print("-" * 52)
for name, text, s in (("full transcript", FULL, full_score),
                      ("last 8 turns", LAST_K, trunc_score)):
    n = approx_tokens(text)
    answered = s * len(PROBE_QUESTIONS)
    print("%-18s %8d %9.0f%% %12s"
          % (name, n, s * 100,
             "%.0f" % (n / answered) if answered else "n/a"))
print()
print("truncation saved %.0f%% of the tokens and lost %.0f points of recall."
      % (100 * (1 - approx_tokens(LAST_K) / approx_tokens(FULL)),
         100 * (full_score - trunc_score)))

memory               tokens     recall tokens/answer
----------------------------------------------------
full transcript         766       100%          128
last 8 turns            142        17%          142

truncation saved 81% of the tokens and lost 83 points of recall.


That trade is the whole subject of this module. Truncation is not wrong - it is
*unpriced*. You saved a large fraction of the tokens and paid for it in facts
you can no longer recall, and nothing in your logs will tell you that happened.

Everything that follows is an attempt to move up and to the left on that
trade-off: keep the recall, drop the tokens.

### 3. The vocabulary

Three words, used precisely for the rest of the module:

- **Working set** - what is actually in the window this turn. Bounded by your
  budget, always.
- **Compression** - replacing many tokens with fewer tokens that carry the same
  decisions. A summary is compression; truncation is *not* (it is deletion).
- **Retention** - the fraction of important facts still recoverable. The number
  the grader above measures.

```
   FULL HISTORY (unbounded, grows forever)
          |
          |  compression        <- summarise, don't delete
          v
   WORKING SET (bounded, fits the budget)
          |
          |  retention          <- the thing you measure
          v
   CAN THE AGENT STILL ANSWER?
```

The mistake to avoid is optimising the working-set size without ever measuring
retention. That is how you ship an agent that is cheap and confidently wrong.

### 4. Which facts survive truncation, specifically?

Averages hide the interesting part. Let us look at *where in the conversation*
each durable fact was stated, versus whether truncation kept it.

In [8]:
print("%-32s %10s %12s" % ("durable fact", "stated at", "in last 8?"))
print("-" * 58)
for label, markers in DURABLE_FACTS:
    where = None
    for i, (role, text) in enumerate(CONVERSATION):
        if any(m in text.lower() for m in markers):
            where = i
            break
    kept = any(m in LAST_K.lower() for m in markers)
    print("%-32s %10s %12s" % (label,
                               "turn %d" % where if where is not None else "?",
                               "yes" if kept else "NO"))
print()
print("Truncation is a filter on TIME, not on IMPORTANCE.")
print("Every hard constraint in this conversation was stated in the first half.")

durable fact                      stated at   in last 8?
----------------------------------------------------------
product name                         turn 8           NO
EU-only hosting                      turn 4           NO
budget 12,000 EUR                   turn 12           NO
price 49 EUR/shop                   turn 16           NO
stack: Postgres + Django            turn 22           NO
no third-party LLM                  turn 26           NO
pilot: Radhaus Krueger, March       turn 32          yes
30-day trial, no free tier          turn 35          yes

Truncation is a filter on TIME, not on IMPORTANCE.
Every hard constraint in this conversation was stated in the first half.


This is not a quirk of our synthetic data - it is how conversations work. People
state their constraints, their name, their budget and their non-negotiables
**early**, then spend the rest of the conversation on details. A recency filter
is therefore systematically biased against exactly the facts you most need.

### 5. Pitfalls

- **Testing memory on short conversations.** Everything works at turn 10. Your
  memory design must be tested at the length you will actually run at.
- **Measuring size without measuring recall.** A smaller working set is not an
  improvement if you cannot say what it cost you.
- **Assuming a bigger window is the fix.** It postpones failure #3 and does
  nothing at all about #1 and #2.
- **Silent loss.** Truncation raises no error and logs nothing. If you do not
  build the grader, you will not learn about the loss from your system - you
  will learn about it from a user.

### Recap

| Idea | Takeaway |
|---|---|
| Three failures | Cost (quadratic), attention, hard limit - in that order |
| Truncation is deletion | Cheap, and it silently removes early facts |
| Recency is biased | Constraints are stated early; recency drops them first |
| Working set / compression / retention | Budgeted context, summarised history, measured recall |

**Next:** [02_rolling_summarization](02_rolling_summarization.ipynb) - the standard
fix, built and graded against the numbers you just produced.